In [28]:
import numpy as np
import pandas as pd

In [29]:
si_data = pd.read_csv("/workspaces/service-data/outputs/snapshots/2025-03-01/utils/si_all.csv", sep=";")

In [30]:
si23 = si_data[si_data["fiscal_yr"] == "2023-2024"].copy()
si22 = si_data[si_data["fiscal_yr"] == "2022-2023"].copy()
siEnd22 = si_data.query("fiscal_yr in ['2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023']").copy()

In [31]:
si23_ext = si23[si23["service_scope_ext_or_ent"]]
si22_ext = si22[si22["service_scope_ext_or_ent"]]

In [32]:
print(f"In 2023-24, there are {si23.shape[0]} services reported, of which {si23_ext.shape[0]} are extern/enterprise, and {si22_ext.shape[0]} services in 2022-23.")

In 2023-24, there are 1715 services reported, of which 1681 are extern/enterprise, and 1690 services in 2022-23.


In [33]:
#si23_ext.count()

In [34]:
si23_ext.columns

Index(['fiscal_yr', 'service_id', 'service_name_en', 'service_name_fr',
       'service_description_en', 'service_description_fr', 'service_type',
       'service_recipient_type', 'service_scope', 'client_target_groups',
       'program_name_en', 'program_name_fr', 'client_feedback_channel',
       'service_fee', 'last_GBA', 'ident_platform', 'ident_platform_comments',
       'os_account_registration', 'os_authentication', 'os_application',
       'os_decision', 'os_issuance', 'os_issue_resolution_feedback',
       'os_comments_client_interaction_en',
       'os_comments_client_interaction_fr',
       'how_has_the_service_been_assessed_for_accessibility',
       'last_service_review', 'last_service_improvement', 'sin_usage',
       'cra_bn_identifier_usage', 'num_phone_enquiries',
       'num_applications_by_phone', 'num_website_visits',
       'num_applications_online', 'num_applications_in_person',
       'num_applications_by_mail', 'num_applications_by_email',
       'num_applicatio

In [35]:
prog_cols = ["fiscal_yr", "service_id", "service_name_en", "program_id", "program_name_en", "org_name_variant", "org_id", "department_en"]

In [36]:
org_names = list(set(si23_ext["org_name_variant"].values))
Nlines = int(len(org_names) / 10)
for k in range(Nlines):
    if k*10 < len(org_names):
        print(", ".join(org_names[k*10:k*10+10]))
print(", ".join(org_names[(Nlines)*10:]))

dnd-mdn, tbs-sct, infc, cannor, prairiescan, pco-bcp, isc-sac, aafc-aac, cfia-acia, pbc-clcc
irb-cisr, cic, statcan, cb-cda, chrc-ccdp, psc-cfp, mpcc-cppm, wage, crtc, nfb-onf
cra-arc, polar-polaire, cihr-irsc, cta-otc, oci-bec, fintrac-canafe, ppsc-sppc, pch, tc, atssc-scdata
nbc-ccbn, dfatd-maecd, cas-satj, aandc-aadnc, vrab-tacra, cbsa-asfc, fcac-acfc, ccohs-cchst, dfo-mpo, iaac-aeic
phac-aspc, nrc-cnrc, jus, fednor, pmprb-cepmb, cgc-ccg, hc-sc, csc-scc, ssc-spc, feddevontario
pacifican, fpcc-cpac, sshrc-crsh, rcmp-grc, cics-scic, csa-asc, nsira-ossnr, cer-rec, pwgsc-tpsgc, fja-cmf
nrcan-rncan, opc-cpvp, csec-cstc, lac-bac, fin, ec, vac-acc, ced-dec, nserc-crsng, acoa-apeca
esdc-edsc, ps-sp, ic, osfi-bsif, csps-efpc, pc, cpc-cpp, oag-bvg


In [37]:
org_ids = list(set(si23_ext["org_id"].values))
Nlines = int(len(org_ids) / 10)
for k in range(Nlines):
    if k*10 < len(org_ids):
        print(" ".join([f"{i:.0f}" for i in org_ids[k*10:k*10+10]]))
print(" ".join([f"{i:.0f}" for i in org_ids[(Nlines)*10:]]))

1 12 26 539 552 46 47 560 561 55
63 65 69 71 74 76 86 93 95 99
110 114 117 118 122 123 124 125 126 127
128 129 130 132 133 134 135 136 137 138
139 140 141 150 151 152 174 199 210 218
221 222 223 227 228 230 237 238 240 246
247 248 253 256 263 266 278 280 282 295
297 302 305 306 313 326 333 348


In [56]:
si_prog = si23_ext.filter(prog_cols).copy()
si_prog.shape

(1681, 8)

In [57]:
si_prog[:4]

,fiscal_yr,service_id,service_name_en,program_id,program_name_en,org_name_variant,org_id,department_en
7249,2023-2024,3596,ATSSC General Inquiries,ISSA2,'Communications Services',atssc-scdata,539.0,Administrative Tribunals Support Service of Ca...
7250,2023-2024,3597,ATSSC Registry Services,BEE01,'Registry Services',atssc-scdata,539.0,Administrative Tribunals Support Service of Ca...
7251,2023-2024,3598,Access to Information and Privacy,ISS02,'Communications Services',atssc-scdata,539.0,Administrative Tribunals Support Service of Ca...
7252,2023-2024,2224,AAFC Contact Centre,BWN02,'Sector Engagement and Development',aafc-aac,1.0,Agriculture and Agri-Food Canada


In [58]:
si_prog["org_id"] = si_prog["org_id"].astype(int)
#si_prog["prog_dept"] = si_prog["program_id"] + "-" + si_prog["org_id"].astype(str)

In [59]:
#si_prog[:4]

In [60]:
# Get the one porgram ID per row as a service can have many programs
si_prog["prog_id_split"] = si_prog["program_id"].str.split(",")
si_prog = si_prog.explode("prog_id_split")
si_prog.shape

(2249, 9)

In [63]:
si_prog["prog_dept"] = si_prog["org_id"].astype(str) + "-" + si_prog["prog_id_split"]
si_prog[:5]

,fiscal_yr,service_id,service_name_en,program_id,program_name_en,org_name_variant,org_id,department_en,prog_id_split,prog_dept
7249,2023-2024,3596,ATSSC General Inquiries,ISSA2,'Communications Services',atssc-scdata,539,Administrative Tribunals Support Service of Ca...,ISSA2,539-ISSA2
7250,2023-2024,3597,ATSSC Registry Services,BEE01,'Registry Services',atssc-scdata,539,Administrative Tribunals Support Service of Ca...,BEE01,539-BEE01
7251,2023-2024,3598,Access to Information and Privacy,ISS02,'Communications Services',atssc-scdata,539,Administrative Tribunals Support Service of Ca...,ISS02,539-ISS02
7252,2023-2024,2224,AAFC Contact Centre,BWN02,'Sector Engagement and Development',aafc-aac,1,Agriculture and Agri-Food Canada,BWN02,1-BWN02
7253,2023-2024,SRV03054,African Swine Fever Industry Preparedness Prog...,BWP13,'African Swine Fever Response',aafc-aac,1,Agriculture and Agri-Food Canada,BWP13,1-BWP13


In [64]:
# Load the spending data from Titan
spending_data = pd.read_csv("/workspaces/service-data/notebooks/svc-spending/program_spending_2024.csv")

In [65]:
print('", "'.join(list(spending_data.columns)))

id", "eternal", "year", "program__id", "program__eternal", "program__activity_code", "program__name_en", "program__name_fr", "cr__id", "cr__eternal", "cr__cr_code", "department__id", "department__eternal", "department__tbs_dept_code", "department__name_en", "department__name_fr", "planned_spending_1", "planned_spending_1_rev", "planned_spending_1_special", "planned_spending_2", "planned_spending_2_rev", "planned_spending_2_special", "planned_spending_3", "planned_spending_3_rev", "planned_spending_3_special", "actual_spending", "planned_FTE_1", "planned_FTE_2", "planned_FTE_3", "actual_FTE", "document__document", "document__year", "program__is_midyear


In [66]:
spending_filtered_cols = ["id", "eternal", "year", "program__id", "program__eternal", "program__activity_code", "program__name_en",
                 "cr__id", "cr__eternal", "cr__cr_code", "department__id", "department__eternal", "department__tbs_dept_code", "department__name_en",
                 "planned_spending_1", "planned_spending_2", "planned_spending_3", "actual_spending",
                 "planned_FTE_1", "planned_FTE_2", "planned_FTE_3", "actual_FTE", "document__document", "document__year", "program__is_midyear"]

In [67]:
spending = spending_data.filter(spending_filtered_cols)

In [68]:
# Get spending published in 2023-24 DRR
spending_24 = spending.query("document__document == 'dp_resources' and document__year == 2024").copy()
spending_24.shape

(1180, 25)

In [69]:
#set(spending_24["document__year"].values)

In [70]:
#spending_24["prog_dept"] = spending_24["program__activity_code"] + "-" + spending_24["department__eternal"].astype(str)

In [74]:
spending_cols = ["id", "eternal", "year", "program__id", "program__eternal", "program__activity_code", "program__name_en", "department__eternal", "department__name_en",
                "planned_spending_1", "planned_spending_2", "planned_spending_3"]

In [75]:
spending_24_skim = spending_24.filter(spending_cols)

In [78]:
#spending_24_skim[:5]

In [79]:
spending_24_skim["prog_dept"] = spending_24_skim["department__eternal"].astype(str) + "-" + spending_24_skim["program__activity_code"]
spending_24_skim[:5]

,id,eternal,year,program__id,program__eternal,program__activity_code,program__name_en,department__eternal,department__name_en,planned_spending_1,planned_spending_2,planned_spending_3,prog_dept
0,30204,9289,2024,5414,249,BXT01,Company Performance,221,Canadian Energy Regulator,12881630.0,12315462.0,12218141.0,221-BXT01
1,30205,9290,2024,5415,834,BXT02,Management System and Industry Performance,221,Canadian Energy Regulator,5302388.0,5169856.0,5143220.0,221-BXT02
2,30206,9291,2024,5416,344,BXT03,Emergency Management,221,Canadian Energy Regulator,1383923.0,1373737.0,1360336.0,221-BXT03
3,30207,9292,2024,5992,1128,BXT04,Regulatory Framework,221,Canadian Energy Regulator,3395017.0,3372312.0,3359798.0,221-BXT04
4,30208,9293,2024,5418,359,BXU01,Energy System Information,221,Canadian Energy Regulator,5993618.0,4241731.0,4210621.0,221-BXU01


In [80]:
# Find the differences in program-department code
diff_code = []
si_progDept = list(si_prog["prog_dept"].unique())
spending_progDept = list(spending_24_skim["prog_dept"].unique())
for i in si_progDept:
    if i not in spending_progDept:
        diff_code.append(i)
len(diff_code)

75

In [83]:
Nlines = int(len(diff_code) / 10)
for k in range(Nlines):
    if k*10 < len(diff_code):
        print("  ".join([i for i in diff_code[k*10:k*10+10]]))
print("  ".join([i for i in diff_code[(Nlines)*10:]]))

539-ISSA2  539-ISS02  141-BLL01  141-ISS02  221-ISS06  221-BXY02  221-ISSA6  46-BRA04  65-BVY02  65-BUG04
74-ISS11  76-ISSA6  86-BSJ03  93-ISS12  99-ISS01  110-BUM02  110-BSJ03  110-ISS02  129-BGV04  129-BTD02
129-BTX05  129-BUV04  129-BWW08  129-BWT04  129-BNQ05  129-BSJ03  128-BGM01  137-BSJ03  137-BGS02  137-BVH06
222-BWN03  561-ISS02  151-BUU01  127-BVG07  63-ISS02  199-BRF01  199-BRA03  223-BSJ03  228-ISSA2  228-BNQ13
228-BNQ16  228-BNQ23  134-BTM01  134-BTO06  134-BTO07  238-ISSA1  238-ISS01  256-BRG01  256-BRG02  253-ISS06
253-ISS07  560-ISS02  263-BVD13  140-ISS02  278-BYI03  248-BRE01  295-BXG06  247-ISS01  247-ISS02  218-BSJ03
138-ISSA1  138-ISSA2  326-BXB05  326-BXB01  326-BXC01  326-BXA04  326-BXB02  326-BXB03  326-BXC06  326-BXC05
326-BXC03  326-BXB06  139-BWI02  139-BWI01  139-BWI15


In [84]:
si_prog[si_prog["prog_dept"].isin(diff_code)].to_csv("incorrect_program_code.csv")